## Сбор и подготовка данных (Data Gathering)

Для проведения исследования были собраны реальные данные с маркетплейса **Wildberries** с использованием библиотеки `requests` и обращением к внутреннему API каталога.

### Формирование выборки
В качестве объекта анализа были выбраны **6 категорий спортивных товаров**, ориентированных на калистенику и воркаут (тренировки с собственным весом):
* Турники
* Брусья
* Гимнастические кольца
* Упоры для отжиманий
* Пояса с цепью для отягощений
* Фитнес-резинки

> **Примечание:** Категории тяжелой атлетики (например, гантели, штанги, грифы) намеренно исключены из исследования, чтобы сохранить однородность выборки в рамках концепции домашнего/воркаут тренинга.

### Техническая реализация парсера
Сбор данных реализован через последовательные GET-запросы к API Wildberries с ограничением глубины парсинга в **15 страниц** для каждой категории (наиболее релевантная поисковая выдача по сортировке `popular`).

Для обеспечения стабильности и отказоустойчивости скрипта внедрены следующие технические решения:
1. **Обработка Rate Limit (Ошибка 429):** При возникновении ошибки `Too Many Requests` активируется алгоритм повторных попыток (до 5 итераций) с нарастающим интервалом ожидания (от 5 секунд).
2. **Имитация поведения пользователя (Throttling):** Для предотвращения блокировок со стороны сервера установлены таймауты: пауза в 5 секунд между страницами и 10 секунд между сменой категорий товаров.
3. **Обогащение данных:** На этапе сбора в словарь каждого объекта динамически добавляется поле `category`. Это необходимо для корректной последующей группировки данных при визуализации.

In [1]:
import time
import requests
import pandas as pd

products_to_find = ['турник', 'пояс с цепью для отягощения', 'упоры для отжиманий', 'фитнес резинки', 'брусья', 'гимнастические кольца']

url = "https://search.wb.ru/exactmatch/ru/common/v18/search"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Accept-Language': 'ru-RU,ru;q=0.9'
}

all_products = []
max_pages = 15

for product in products_to_find:
    for page in range(1, max_pages + 1):
        
        params = {
            'appType': '1',
            'curr': 'rub',
            'dest': '-1257786',
            'query': product,
            'resultset': 'catalog',
            'sort': 'popular',
            'spp': '30',
            'page': str(page)
        }
        
        max_retries = 5
        retry_delay = 5
        page_products = []
        
        for attempt in range(max_retries):
            response = requests.get(url, params=params, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                page_products = data.get('products', [])
                for page_product in page_products:
                    page_product['category'] = product
                break
                
            elif response.status_code == 429:
                wait_seconds = retry_delay * (attempt + 1)
                print(f"Попытка {attempt + 1}: Ошибка 429. Ждем {wait_seconds} сек")
                time.sleep(wait_seconds)
            else:
                print(f"Ошибка {response.status_code}. Пропускаем страницу")
                break
        
        if not page_products:
            print("Товары закончились или произошла ошибка. Прерываем цикл постраничного сбора")
            break
            
        all_products.extend(page_products)
        print(f"На странице {page} найдено товаров: {len(page_products)} для категории: '{product}'")
        
        time.sleep(5)
    time.sleep(10)

print(f"\nСбор завершен. Всего собрано товаров: {len(all_products)}")


На странице 1 найдено товаров: 100 для категории: 'турник'


KeyboardInterrupt: 

## Инициализация датафрейма и сохранение данных

После завершения парсинга сырые данные в формате JSON конвертируются в табличный вид для дальнейшего анализа.

In [ ]:
df = pd.DataFrame(all_products)
df.to_csv('WorkoutProducts.csv')

category
упоры для отжиманий            1500
фитнес резинки                 1500
брусья                         1500
гимнастические кольца          1500
турник                         1200
пояс с цепью для отягощения    1100
Name: count, dtype: int64

### Первичная валидация выборки
Для контроля корректности сбора данных проверим распределение количества найденных товаров по каждой целевой категории. Это позволит убедиться, что API маркетплейса вернуло данные по всем запросам в полном объеме.

In [1]:
df['category'].value_counts()

NameError: name 'df' is not defined